<a href="https://colab.research.google.com/github/Breyzz/dti_predictor/blob/main/dti_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 DTI Predictor — Part 3: Production Model

**The portfolio version.** Four major upgrades over Part 2:

| Component | Part 2 | Part 3 |
|---|---|---|
| Protein encoder | ESM-2 150M (640-dim) | ESM-2 650M (1280-dim) |
| Drug encoder | GCN | GAT (Graph Attention) |
| Training data | Davis | Davis + KIBA |
| Tasks | Single (classification) | Multi-task with shared encoder |

---

### Why each change matters

**ESM-2 650M:** 33 transformer layers vs 30, trained on more data. Standard backbone in published DTI papers.

**GAT over GCN:** GCN averages neighbours equally. GAT learns attention weights — letting the model focus on the carbonyl oxygen of an amide, for example, rather than treating it the same as the methyl group next to it. This matches the chemistry.

**Davis + KIBA multi-task:** KIBA has ~118k pairs (vs Davis's 30k) and uses a different affinity score. Training on both with a shared encoder forces the model to learn *transferable* drug/protein features instead of dataset-specific quirks.

**Output strategy:** This notebook saves three artifacts that the Streamlit app will consume:
1. `final_model.pt` — trained weights
2. `esm_cache_650m.pt` — pre-computed protein embeddings (so the app doesn't need to load ESM-2 itself)
3. `model_metadata.json` — config (atom features dim, ESM dim, etc.)

---
## Section 0 — Setup

⚠️ **GPU memory check:** ESM-2 650M needs ~3GB just sitting in GPU. T4 has 15GB so we're fine, but close other notebooks first.

In [ ]:
# 0.1 — Install dependencies
import subprocess, warnings
warnings.filterwarnings('ignore')

def run(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-1000:])

print("Installing packages... (~3 min)")
run("pip install -q rdkit biopython tqdm matplotlib seaborn scikit-learn pandas")
run("pip install -q torch-geometric")
run("pip install -q fair-esm")
print("✓ Done")

Installing packages... (~3 min)
✓ Done


In [ ]:
# 0.2 — Imports
import torch, json, urllib.request, gc
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, global_add_pool
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
import esm
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA = Path('data');  DATA.mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)
Path('artifacts').mkdir(exist_ok=True)  # final outputs go here

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
Memory : 15.6 GB


---
## Section 1 — Featurization helpers

In [ ]:
# 1 — Re-define molecular featurization
ATOM_TYPES = ['C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca','Fe',
              'As','Al','I','B','V','K','Tl','Yb','Sb','Sn','Ag','Pd','Co',
              'Se','Ti','Zn','H','Li','Ge','Cu','Au','Ni','Cd','In','Mn',
              'Zr','Cr','Pt','Hg','Pb']
HYBRID_TYPES = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                Chem.rdchem.HybridizationType.SP3D2]
NODE_DIM = 74  # locked in from Part 1

def one_hot(value, choices):
    enc = [0] * (len(choices) + 1)
    try: enc[choices.index(value)] = 1
    except ValueError: enc[-1] = 1
    return enc

def atom_features(atom):
    try: impl_v = atom.GetValence(Chem.ValenceType.IMPLICIT)
    except: impl_v = atom.GetImplicitValence()
    return (one_hot(atom.GetSymbol(), ATOM_TYPES) +
            one_hot(atom.GetDegree(), list(range(11))) +
            one_hot(impl_v, list(range(6))) +
            [atom.GetFormalCharge(), atom.GetTotalNumHs(), atom.GetNumRadicalElectrons()] +
            one_hot(atom.GetHybridization(), HYBRID_TYPES) +
            [int(atom.GetIsAromatic()), int(atom.IsInRing())])

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        ei += [[i,j],[j,i]]
    if not ei:
        return Data(x=x, edge_index=torch.zeros((2,1),dtype=torch.long))
    return Data(x=x, edge_index=torch.tensor(ei,dtype=torch.long).t().contiguous())

print(f"✓ Helpers ready (NODE_DIM = {NODE_DIM})")

---
## Section 2 — Load both datasets (Davis + KIBA)

**Davis:** 68 drugs × 442 kinases = ~30k pairs. Affinity in nM, threshold Kd < 30 nM.

**KIBA:** 2,111 drugs × 229 kinases = ~118k pairs. KIBA score combines Ki/Kd/IC50 — threshold ≥ 12.1 (higher = stronger binder, opposite direction from Kd).

We'll harmonize both as binary classification + regression.

In [ ]:
# 2.1 — Download Davis
BASE_DAVIS = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/davis'
for fname in ['ligands_can.txt', 'proteins.txt', 'Y']:
    p = DATA / f'davis_{fname}'
    if not p.exists():
        urllib.request.urlretrieve(f'{BASE_DAVIS}/{fname}', p)

davis_ligands  = json.load(open(DATA / 'davis_ligands_can.txt'))
davis_proteins = json.load(open(DATA / 'davis_proteins.txt'))
davis_Y        = np.load(DATA / 'davis_Y', allow_pickle=True, encoding='latin1')

print(f"Davis  drugs: {len(davis_ligands):>6}  proteins: {len(davis_proteins):>4}  matrix: {davis_Y.shape}")

In [ ]:
# 2.2 — Download KIBA
BASE_KIBA = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/kiba'
for fname in ['ligands_can.txt', 'proteins.txt', 'Y']:
    p = DATA / f'kiba_{fname}'
    if not p.exists():
        urllib.request.urlretrieve(f'{BASE_KIBA}/{fname}', p)

kiba_ligands  = json.load(open(DATA / 'kiba_ligands_can.txt'))
kiba_proteins = json.load(open(DATA / 'kiba_proteins.txt'))
kiba_Y        = np.load(DATA / 'kiba_Y', allow_pickle=True, encoding='latin1')

print(f"KIBA   drugs: {len(kiba_ligands):>6}  proteins: {len(kiba_proteins):>4}  matrix: {kiba_Y.shape}")

# Combined unique proteins for ESM-2 caching
all_proteins = {**davis_proteins, **kiba_proteins}
print(f"\nUnique proteins across both datasets: {len(all_proteins):,}")

In [ ]:
# 2.3 — Build harmonized pair tables
# Davis: log-transform Kd to pKd, threshold pKd >= 7 (= Kd <= 100 nM, slightly looser than Davis's 30 nM
# to get more positives). KIBA: threshold KIBA score >= 12.1 (standard).

DAVIS_PKD_THRESHOLD = 7.0   # pKd = -log10(Kd in M); Kd 100nM = pKd 7
KIBA_THRESHOLD       = 12.1

def davis_to_pkd(kd_nm):
    """Convert Kd in nM to pKd. Higher pKd = stronger binding."""
    return -np.log10(kd_nm * 1e-9)

# Davis table
davis_rows = []
drug_names = list(davis_ligands.keys())
prot_names = list(davis_proteins.keys())
for i, dn in enumerate(drug_names):
    for j, pn in enumerate(prot_names):
        v = davis_Y[i, j]
        if not np.isnan(v):
            pkd = davis_to_pkd(v)
            davis_rows.append({
                'drug': dn, 'smiles': davis_ligands[dn],
                'protein': pn, 'sequence': davis_proteins[pn],
                'affinity': pkd,                       # regression target (higher = stronger)
                'label': int(pkd >= DAVIS_PKD_THRESHOLD),
                'dataset': 'davis',
            })
davis_df = pd.DataFrame(davis_rows)

# KIBA table
kiba_rows = []
k_drug_names = list(kiba_ligands.keys())
k_prot_names = list(kiba_proteins.keys())
for i, dn in enumerate(k_drug_names):
    for j, pn in enumerate(k_prot_names):
        v = kiba_Y[i, j]
        if not np.isnan(v):
            kiba_rows.append({
                'drug': dn, 'smiles': kiba_ligands[dn],
                'protein': pn, 'sequence': kiba_proteins[pn],
                'affinity': v,                         # already an affinity score, higher = stronger
                'label': int(v >= KIBA_THRESHOLD),
                'dataset': 'kiba',
            })
kiba_df = pd.DataFrame(kiba_rows)

# Concatenate and shuffle
combined = pd.concat([davis_df, kiba_df], ignore_index=True)
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset summary")
print("─" * 50)
for ds in ['davis', 'kiba']:
    sub = combined[combined.dataset == ds]
    pos = sub.label.sum()
    print(f"{ds:<8} pairs: {len(sub):>7,}   positive: {pos:>6,} ({100*pos/len(sub):.1f}%)")
print(f"{'total':<8} pairs: {len(combined):>7,}   positive: {combined.label.sum():>6,} ({100*combined.label.sum()/len(combined):.1f}%)")

---
## Section 3 — Embed Proteins with ESM-2 650M

**Strategy:** Embed each unique protein **once** and cache to disk. Then load cache without keeping ESM-2 in memory during training. This avoids running out of GPU memory.

**ESM-2 650M size:** 33 layers, 1280-dim embeddings, ~2.5GB model on disk. Embedding 442 + 229 = ~470 unique proteins takes ~10 min on T4.

In [ ]:
# 3.1 — Load ESM-2 650M
ESM_CACHE = DATA / 'esm_cache_650m.pt'
ESM_LAYERS = 33
ESM_DIM    = 1280

if ESM_CACHE.exists():
    print('Found existing cache, skipping ESM-2 load.')
    esm_cache = torch.load(ESM_CACHE, weights_only=False)
    print(f'✓ Loaded {len(esm_cache)} cached embeddings')
else:
    print('Loading ESM-2 650M (~2.5GB, takes a while)...')
    esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    esm_model = esm_model.to(DEVICE).eval()
    batch_converter = alphabet.get_batch_converter()
    print(f'✓ Loaded — params: {sum(p.numel() for p in esm_model.parameters()):,}')

In [ ]:
# 3.2 — Embed all unique proteins (skip if cache exists)
if not ESM_CACHE.exists():
    @torch.no_grad()
    def embed_protein(seq, max_len=1022):
        seq = seq[:max_len]
        _, _, tokens = batch_converter([('p', seq)])
        tokens = tokens.to(DEVICE)
        out = esm_model(tokens, repr_layers=[ESM_LAYERS])
        reps = out['representations'][ESM_LAYERS]
        return reps[0, 1:len(seq)+1].mean(dim=0).cpu()

    esm_cache = {}
    for name, seq in tqdm(all_proteins.items(), desc='ESM-2 embedding'):
        esm_cache[name] = embed_protein(seq)

    torch.save(esm_cache, ESM_CACHE)
    print(f'\n💾 Saved {len(esm_cache)} embeddings to {ESM_CACHE}')

    # Free up GPU memory
    del esm_model, batch_converter, alphabet
    gc.collect(); torch.cuda.empty_cache()
    print('✓ Freed ESM-2 from GPU memory')

first_name = next(iter(esm_cache))
print(f'\nExample: {first_name} → shape {esm_cache[first_name].shape}')

---
## Section 4 — Build the Multi-Task Dataset

Each training example carries:
- **Drug graph** (PyG Data)
- **Protein embedding** (1280-dim ESM-2 vector)
- **Binary label** (0/1 for classification)
- **Affinity value** (continuous for regression — pKd for Davis, KIBA score for KIBA)
- **Dataset tag** (so we know which regression head to use)

In [ ]:
# 4 — Build samples
FEATURES_PATH = DATA / 'features_multitask.pt'

if FEATURES_PATH.exists():
    samples = torch.load(FEATURES_PATH, weights_only=False)
    print(f'✓ Loaded {len(samples):,} cached samples')
else:
    samples = []
    skipped = 0
    DATASET_IDS = {'davis': 0, 'kiba': 1}

    for _, row in tqdm(combined.iterrows(), total=len(combined), desc='Building'):
        g = smiles_to_graph(row['smiles'])
        if g is None: skipped += 1; continue
        prot_emb = esm_cache[row['protein']]
        samples.append({
            'graph':    g,
            'prot_emb': prot_emb,
            'label':    torch.tensor(row['label'],    dtype=torch.float),
            'affinity': torch.tensor(row['affinity'], dtype=torch.float),
            'dataset':  torch.tensor(DATASET_IDS[row['dataset']], dtype=torch.long),
        })
    print(f'\n✓ Built {len(samples):,}  |  Skipped {skipped}')
    torch.save(samples, FEATURES_PATH)
    print(f'💾 Saved to {FEATURES_PATH}')

---
## Section 5 — GAT-based Multi-Task Model

**Architecture changes from Part 2:**

```
           ╭─ Multi-head attention (4 heads) ─╮
  Atom →   │  GATConv → GATConv → GATConv      │  → drug embedding (256)
           ╰───────────────────────────────────╯              │
  Protein → ESM-2 (1280) → MLP                 → protein emb (256)
                                                                ↓
                                                       [shared encoder fusion]
                                                                ↓
                                                            ┌───┴───┐
                                                  ┌─ classifier ──┐
                                                  ├─ regressor (Davis pKd)
                                                  └─ regressor (KIBA score)
```

**Why three heads?** Davis pKd and KIBA score are on different scales — one regressor can't fit both. Having dataset-specific regression heads lets the model learn shared *features* in the encoder while specializing the *outputs* per dataset.

**Auxiliary loss:** Adding regression even though we mainly care about classification gives the model more learning signal — predicting an exact affinity is harder than predicting binary, so the encoder learns richer features.

In [ ]:
# 5.1 — GAT Drug Encoder
class GATDrugEncoder(nn.Module):
    """3 GATConv layers with multi-head attention + residuals + global pool."""

    def __init__(self, node_dim=NODE_DIM, hidden=128, heads=4, out=256, layers=3, drop=0.2):
        super().__init__()
        self.drop = drop
        self.proj = nn.Linear(node_dim, hidden)

        # GAT layers — concat=False averages heads back to `hidden` dim
        self.gats = nn.ModuleList([
            GATConv(hidden, hidden, heads=heads, concat=False, dropout=drop)
            for _ in range(layers)
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(layers)])

        # Global pool: combine mean + sum for richer representation
        self.out = nn.Sequential(
            nn.Linear(hidden * 2, out),  # *2 because we concat mean + sum pooling
            nn.ReLU(), nn.Dropout(drop),
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.proj(x))
        for gat, bn in zip(self.gats, self.bns):
            x_new = F.elu(bn(gat(x, edge_index)))
            x_new = F.dropout(x_new, p=self.drop, training=self.training)
            x = x + x_new  # residual

        # Combine multiple pooling strategies
        pooled = torch.cat([
            global_mean_pool(x, batch),
            global_add_pool(x, batch),
        ], dim=1)
        return self.out(pooled)


class ESMProteinEncoder(nn.Module):
    def __init__(self, in_dim=ESM_DIM, out=256, drop=0.2):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, 768), nn.ReLU(), nn.BatchNorm1d(768), nn.Dropout(drop),
            nn.Linear(768, 384),    nn.ReLU(), nn.BatchNorm1d(384), nn.Dropout(drop),
            nn.Linear(384, out),    nn.ReLU(), nn.BatchNorm1d(out),
        )
    def forward(self, x):
        return self.enc(x)


class MultiTaskDTI(nn.Module):
    """Shared encoder + 3 heads (classification, Davis regression, KIBA regression)."""

    def __init__(self, node_dim=NODE_DIM, esm_dim=ESM_DIM,
                 drug_emb=256, prot_emb=256, drop=0.2):
        super().__init__()
        self.drug_enc = GATDrugEncoder(node_dim=node_dim, out=drug_emb, drop=drop)
        self.prot_enc = ESMProteinEncoder(in_dim=esm_dim, out=prot_emb, drop=drop)

        fd = drug_emb + prot_emb  # 512
        self.shared = nn.Sequential(
            nn.Linear(fd, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(drop),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop),
        )

        # Three task heads
        self.cls_head     = nn.Linear(128, 1)  # binary interaction
        self.reg_davis    = nn.Linear(128, 1)  # pKd
        self.reg_kiba     = nn.Linear(128, 1)  # KIBA score

    def forward(self, drug, prot):
        d = self.drug_enc(drug.x, drug.edge_index, drug.batch)
        p = self.prot_enc(prot)
        h = self.shared(torch.cat([d, p], dim=1))
        return {
            'logit':     self.cls_head(h).squeeze(-1),
            'reg_davis': self.reg_davis(h).squeeze(-1),
            'reg_kiba':  self.reg_kiba(h).squeeze(-1),
        }


model = MultiTaskDTI()
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MultiTaskDTI parameters: {params:,}")
print(f"\nComponents:")
print(f"  GATDrugEncoder    : 3-layer GAT with 4 heads, mean+sum pool → 256-dim")
print(f"  ESMProteinEncoder : 1280 → 768 → 384 → 256")
print(f"  Shared trunk      : 512 → 256 → 128")
print(f"  3 output heads    : classification + Davis pKd + KIBA score")

---
## Section 6 — Multi-Task Training

**Total loss = classification BCE + λ_reg × regression MSE**, where regression loss only applies to samples from the matching dataset.

In [ ]:
# 6.1 — DataLoaders
class DTIDataset(Dataset):
    def __init__(self, s): self.s = s
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return self.s[i]

def collate(batch):
    graphs   = [b['graph'] for b in batch]
    prot     = torch.stack([b['prot_emb'] for b in batch])
    label    = torch.stack([b['label'] for b in batch])
    affinity = torch.stack([b['affinity'] for b in batch])
    dataset  = torch.stack([b['dataset'] for b in batch])
    return Batch.from_data_list(graphs), prot, label, affinity, dataset

labels_all = [int(s['label'].item()) for s in samples]
tr_idx, val_idx = train_test_split(range(len(samples)), test_size=0.15,
                                   stratify=labels_all, random_state=42)

BATCH = 64
train_loader = DataLoader(DTIDataset([samples[i] for i in tr_idx]),  batch_size=BATCH, shuffle=True,  collate_fn=collate, num_workers=0)
val_loader   = DataLoader(DTIDataset([samples[i] for i in val_idx]), batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=0)

print(f"Train: {len(tr_idx):,} pairs ({len(train_loader)} batches)")
print(f"Val:   {len(val_idx):,} pairs ({len(val_loader)} batches)")

In [ ]:
# 6.2 — Multi-task training loop
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS    = 30      # KIBA's ~118k pairs makes each epoch slower
LR        = 1e-3
REG_LAMBDA = 0.3    # weight on regression auxiliary loss

model = MultiTaskDTI().to(DEVICE)

# Class weighting for imbalance
pos_count = sum(labels_all)
pw = torch.tensor([(len(labels_all) - pos_count) / pos_count], dtype=torch.float).to(DEVICE)
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pw)
mse_loss = nn.MSELoss(reduction='none')

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

DATASET_DAVIS, DATASET_KIBA = 0, 1

def epoch_pass(loader, train=True):
    model.train() if train else model.eval()
    tot_loss, tot_cls_loss, tot_reg_loss = 0, 0, 0
    preds, labs = [], []
    n = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for db, pb, lb, af, ds in loader:
            db, pb, lb, af, ds = db.to(DEVICE), pb.to(DEVICE), lb.to(DEVICE), af.to(DEVICE), ds.to(DEVICE)
            out = model(db, pb)

            # Classification loss (all samples)
            cls_l = bce_loss(out['logit'], lb)

            # Regression loss (per-dataset)
            davis_mask = (ds == DATASET_DAVIS)
            kiba_mask  = (ds == DATASET_KIBA)

            reg_l = torch.tensor(0.0, device=DEVICE)
            if davis_mask.any():
                reg_l = reg_l + mse_loss(out['reg_davis'][davis_mask], af[davis_mask]).mean()
            if kiba_mask.any():
                reg_l = reg_l + mse_loss(out['reg_kiba'][kiba_mask],  af[kiba_mask]).mean()

            loss = cls_l + REG_LAMBDA * reg_l

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            bs = lb.size(0)
            tot_loss     += loss.item()    * bs
            tot_cls_loss += cls_l.item()   * bs
            tot_reg_loss += reg_l.item()   * bs
            n += bs
            preds.extend(torch.sigmoid(out['logit']).detach().cpu().numpy())
            labs.extend(lb.cpu().numpy())

    pa = np.array(preds)
    return {
        'loss':     tot_loss / n,
        'cls_loss': tot_cls_loss / n,
        'reg_loss': tot_reg_loss / n,
        'auc':      roc_auc_score(labs, pa),
        'acc':      accuracy_score(labs, (pa > 0.5).astype(int)),
    }

print(f"Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"pos_weight : {pw.item():.2f}")
print(f"REG_LAMBDA : {REG_LAMBDA}")
print()

best_auc = 0
history = []

for ep in range(1, EPOCHS+1):
    tr = epoch_pass(train_loader, train=True)
    vl = epoch_pass(val_loader,   train=False)
    scheduler.step()
    history.append({'epoch': ep, **{f'tr_{k}':v for k,v in tr.items()}, **{f'vl_{k}':v for k,v in vl.items()}})

    flag = ''
    if vl['auc'] > best_auc:
        best_auc = vl['auc']
        torch.save(model.state_dict(), 'checkpoints/best_multitask.pt')
        flag = '  ✓ best'

    print(f"Ep {ep:2d}/{EPOCHS}  "
          f"tr[loss={tr['loss']:.4f} auc={tr['auc']:.4f}]  "
          f"vl[loss={vl['loss']:.4f} cls={vl['cls_loss']:.4f} reg={vl['reg_loss']:.4f} auc={vl['auc']:.4f} acc={vl['acc']:.4f}]{flag}")

print(f"\n🎯 Best validation AUC-ROC: {best_auc:.4f}")
print(f"   (Compare to Part 2 baseline ~0.85, Part 2 ESM-2 ~0.88)")

In [ ]:
# 6.3 — Plot training curves
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (key, title) in zip(axes, [('loss', 'Loss'), ('auc', 'AUC-ROC'), ('acc', 'Accuracy')]):
    ax.plot(hist_df['epoch'], hist_df[f'tr_{key}'], label='Train', color='#3498db', lw=1.5)
    ax.plot(hist_df['epoch'], hist_df[f'vl_{key}'], label='Val',   color='#e74c3c', lw=1.5)
    ax.set_xlabel('Epoch'); ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Multi-Task Training (Davis + KIBA)', y=1.02)
plt.tight_layout()
plt.savefig('artifacts/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Per-Dataset Evaluation

Let's see how well the model does on each dataset separately.

In [ ]:
# 7 — Evaluate on Davis-only and KIBA-only val splits
model.load_state_dict(torch.load('checkpoints/best_multitask.pt', map_location=DEVICE, weights_only=True))
model.eval()

@torch.no_grad()
def evaluate_subset(dataset_id, name):
    subset = [samples[i] for i in val_idx if samples[i]['dataset'].item() == dataset_id]
    if not subset:
        print(f'{name}: no val samples'); return

    loader = DataLoader(DTIDataset(subset), batch_size=BATCH, shuffle=False, collate_fn=collate)
    preds, labs, regs, afs = [], [], [], []

    for db, pb, lb, af, ds in loader:
        db, pb = db.to(DEVICE), pb.to(DEVICE)
        out = model(db, pb)
        preds.extend(torch.sigmoid(out['logit']).cpu().numpy())
        labs.extend(lb.numpy())
        reg_key = 'reg_davis' if dataset_id == DATASET_DAVIS else 'reg_kiba'
        regs.extend(out[reg_key].cpu().numpy())
        afs.extend(af.numpy())

    pa = np.array(preds)
    auc  = roc_auc_score(labs, pa)
    acc  = accuracy_score(labs, (pa > 0.5).astype(int))
    rmse = np.sqrt(mean_squared_error(afs, regs))
    print(f'  {name:<8}  AUC: {auc:.4f}   Acc: {acc:.4f}   Regression RMSE: {rmse:.4f}')
    return auc, acc, rmse

print('Per-dataset validation performance')
print('─' * 60)
evaluate_subset(DATASET_DAVIS, 'Davis')
evaluate_subset(DATASET_KIBA, 'KIBA')

---
## Section 8 — Export Artifacts for the Streamlit App

We save:
1. **Model weights** (the `.pt` file)
2. **ESM-2 protein cache** (so the app doesn't need to load ESM-2)
3. **Drug library** (Davis + KIBA SMILES) — for virtual screening UI
4. **Metadata JSON** (model dims, dataset names, etc.)

In [ ]:
# 8 — Save deployment artifacts
import shutil

ART = Path('artifacts')

# 1. Model weights
shutil.copy('checkpoints/best_multitask.pt', ART / 'final_model.pt')

# 2. Protein embedding cache
shutil.copy(ESM_CACHE, ART / 'esm_cache.pt')

# 3. Drug library — combine Davis + KIBA SMILES
drug_library = {}
for name, smi in davis_ligands.items():
    drug_library[name] = {'smiles': smi, 'source': 'davis'}
for name, smi in kiba_ligands.items():
    if name not in drug_library:  # Davis takes priority on duplicates
        drug_library[name] = {'smiles': smi, 'source': 'kiba'}

with open(ART / 'drug_library.json', 'w') as f:
    json.dump(drug_library, f, indent=2)

# 4. Protein library (for screening UI dropdown)
protein_library = {name: {'sequence': seq, 'length': len(seq),
                          'source': 'davis' if name in davis_proteins else 'kiba'}
                   for name, seq in all_proteins.items()}

with open(ART / 'protein_library.json', 'w') as f:
    json.dump(protein_library, f, indent=2)

# 5. Metadata
metadata = {
    'node_dim':     NODE_DIM,
    'esm_dim':      ESM_DIM,
    'esm_layers':   ESM_LAYERS,
    'esm_model':    'esm2_t33_650M_UR50D',
    'drug_emb':     256,
    'prot_emb':     256,
    'datasets':     ['davis', 'kiba'],
    'best_val_auc': float(best_auc),
    'epochs':       EPOCHS,
    'davis_threshold_pkd': DAVIS_PKD_THRESHOLD,
    'kiba_threshold':       KIBA_THRESHOLD,
    'num_proteins': len(esm_cache),
    'num_drugs':    len(drug_library),
}
with open(ART / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved to artifacts/:')
for p in sorted(ART.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:<28} {size_mb:>8.1f} MB')

total = sum(p.stat().st_size for p in ART.iterdir()) / 1e6
print(f'\nTotal: {total:.1f} MB')

In [ ]:
# 8.1 — Download artifacts to your computer
# Run this in Colab to download the artifacts folder as a zip
shutil.make_archive('dti_artifacts', 'zip', 'artifacts')

from google.colab import files
files.download('dti_artifacts.zip')
print('✓ Download started — save next to your Streamlit app')

---
## Done! 🎉

Your `artifacts/` folder now contains everything the Streamlit app needs:

| File | Purpose |
|---|---|
| `final_model.pt` | Trained weights |
| `esm_cache.pt` | Pre-computed protein embeddings (~50MB) |
| `drug_library.json` | All Davis+KIBA drug SMILES |
| `protein_library.json` | All Davis+KIBA protein sequences |
| `model_metadata.json` | Architecture config |
| `training_curves.png` | Loss/AUC curves |

**Next:** Set up the Streamlit app locally with these artifacts (instructions in `README.md`).